# Tutorial 5: PyMC Surrogate Learning (`pymc_gp`)

Estimated time: 30-50 minutes

## Prerequisites
`pymc` and `arviz` installed. Run `mm doctor` to verify.

## Learning aims
- Fit/evaluate surrogate models through the CLI and inspect artifacts
- Build intuition for prior, posterior, and posterior predictive uncertainty
- Train a multi-output joint surrogate (`output_correlation: "full"`)


In [ ]:
# Cross-platform setup — works on Windows / macOS / Linux.
# Self-contained: walks up to find the repo root, adds src/ to sys.path,
# then imports the official bootstrap helper for chdir + later run_cli use.
import os, sys
from pathlib import Path

_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    _marker = _candidate / 'pyproject.toml'
    if _marker.is_file() and 'name = "metamodeler"' in _marker.read_text():
        ROOT = _candidate
        break
else:
    raise FileNotFoundError('Could not locate metamodeler repo root from ' + str(_here))

_src = str((ROOT / 'src').resolve())
if _src not in sys.path:
    sys.path.insert(0, _src)
if Path.cwd().resolve() != ROOT.resolve():
    os.chdir(ROOT)

from metamodeler.tutorial import bootstrap, run_cli  # noqa: E402
ROOT = bootstrap()
print('Repo root:', ROOT)


## Step 1: Ensure training dataset exists

In [ ]:
run_cli('run', 'tutorials/specs/model.toy.grid.json')


## Step 2: Fit single-output surrogate and list artifacts

In [ ]:
run_cli('surrogate', 'fit', 'tutorials/specs/surrogate.toy.pymc_gp.json')
run_cli('surrogate', 'list')


## Step 3: Evaluate on new inputs

In [ ]:
import json
from metamodeler.spec import SurrogateSpec
from metamodeler.surrogates import eval_surrogate

spec_payload = json.loads(
    (ROOT / 'tutorials/specs/surrogate.toy.pymc_gp.json').read_text()
)
spec = SurrogateSpec.model_validate(spec_payload)
result = eval_surrogate(
    spec=spec,
    inputs_payload={'a': [0.25, 0.75, 1.25, 1.75], 'b': [0.2, 0.6, 1.0, 1.4]},
    n=200,
)
print('sample shape:', result['sample_shape'])
print('summary keys:', list(result['summary'].keys()))


## Step 4: Plot predictive mean ± std (graphic)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

out_name = spec.outputs[0]
mean = np.asarray(result['summary']['mean'][out_name], dtype=float)
std = np.asarray(result['summary']['std'][out_name], dtype=float)
x = np.arange(len(mean))

plt.figure(figsize=(6, 4))
plt.errorbar(x, mean, yerr=std, fmt='o-', capsize=4)
plt.title(f'PyMC surrogate predictive {out_name}: mean +/- std')
plt.xlabel('query point index')
plt.ylabel(f'predicted {out_name}')
plt.grid(True, alpha=0.3)
plt.show()


## Step 5: Multi-output joint surrogate (optional)
Train a 2-output surrogate that learns the joint distribution of two outputs.
Requires a multi-output dataset; we synthesize one from the toy model.


In [ ]:
import json
from pathlib import Path
import numpy as np

store = ROOT / 'tmp/tutorial_multi_output_store'
(store / 'runs').mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(0)
for idx in range(80):
    a = float(rng.uniform(-2, 2))
    b = float(rng.uniform(-1, 1))
    y1 = 1.7 * a - 0.8 * b + 0.2 + float(rng.normal(0, 0.05))
    y2 = -0.5 * a + 1.2 * b - 0.3 + float(rng.normal(0, 0.05))
    run_dir = store / 'runs' / f'run_{idx:03d}'
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / 'inputs.json').write_text(json.dumps({'a': a, 'b': b}))
    (run_dir / 'outputs.json').write_text(json.dumps({'y1': y1, 'y2': y2}))
print('Synthesized', 80, 'runs.')
run_cli('surrogate', 'fit', 'examples/surrogates/surrogate.toy.multi_output.json')


## Scientific mini-lesson
- Prior: beliefs before data.
- Posterior: updated beliefs after data.
- Posterior predictive: uncertainty-aware output prediction.
- For multi-output joint surrogates, `output_correlation: "full"` learns the cross-output covariance.
  Use `"diagonal"` (default) for independent per-output models.


In [ ]:
# Optional: run the dedicated PyMC backend test as a sanity check.
result = run_cli(
    'surrogate', 'list', capture=True,
)
print(result.stdout)
